# Section 2: Publish-Subscribe Communication Protocol

This section implements the **publish-subscribe (pub-sub) communication protocol** for a MetaGPT-inspired multi-agent software development pipeline.

**Key Idea:** Instead of agents passing data directly to each other in a rigid chain, agents communicate through a shared **Message Bus**. Each agent:
- **Subscribes** to topics it needs input from
- **Publishes** its output to a topic
- Remains **decoupled** from other agents — it doesn't know who produces or consumes its messages

This mirrors MetaGPT's approach where role-based agents share a structured communication environment, enabling flexible, event-driven collaboration.

---

### Topic Map

| Agent | Subscribes To | Publishes To |
|---|---|---|
| ProductManager | `user_requirements` | `prd_output` |
| Architect | `prd_output` | `system_design` |
| ProjectManager | `system_design` | `task_breakdown` |
| Engineer | `task_breakdown`, `prd_output`, `system_design` | `code_output` |
| QAEngineer | `code_output`, `prd_output` | `test_output` |

---
## Cell 1 — Imports, API Key & Output Schemas
Shared foundation: imports, API key, and the same output dataclasses used across all three sections.

In [19]:
# ============================================================
# CELL 1: Imports, API Key & Output Schemas
# ============================================================

import json
import requests
import uuid
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
import dotenv


# --- Shared Output Schemas (same as Section 1) ---
API_KEY = "Use your api keys"


@dataclass
class PRDDocument:
    original_requirements: str = ""
    product_goals: list = field(default_factory=list)
    user_stories: list = field(default_factory=list)
    competitive_analysis: list = field(default_factory=list)
    requirement_analysis: str = ""
    requirement_pool: list = field(default_factory=list)
    ui_design_draft: str = ""

@dataclass
class SystemDesign:
    implementation_approach: str = ""
    file_list: list = field(default_factory=list)
    data_structures: str = ""
    interface_definitions: str = ""
    program_call_flow: str = ""

@dataclass
class TaskList:
    required_packages: list = field(default_factory=list)
    task_list: list = field(default_factory=list)
    shared_knowledge: str = ""
    logic_analysis: list = field(default_factory=list)

@dataclass
class CodeOutput:
    filename: str = ""
    code: str = ""
    dependencies: list = field(default_factory=list)

@dataclass
class TestOutput:
    test_filename: str = ""
    test_code: str = ""
    review_notes: str = ""

print("Imports, API key, and schemas ready.")

Imports, API key, and schemas ready.


---
## Cell 2 — Prompt Templates & LLM Client
Shared prompt templates for each role and the HTTP-based LLM client using OpenAI API.

In [20]:
# ============================================================
# CELL 2: Prompt Templates & LLM Client
# ============================================================

PRODUCT_MANAGER_PROMPT = """
You are a Product Manager. Given the user requirement below,
produce a Product Requirement Document in EXACTLY this JSON format:

{{
    "original_requirements": "<restate the requirement>",
    "product_goals": ["goal1", "goal2", "goal3"],
    "user_stories": ["story1", "story2", "story3"],
    "competitive_analysis": ["competitor1: analysis", "competitor2: analysis"],
    "requirement_analysis": "<detailed analysis>",
    "requirement_pool": [
        ["<requirement description>", "P0"],
        ["<requirement description>", "P1"]
    ],
    "ui_design_draft": "<description of UI layout>"
}}

User Requirement: {user_requirement}

Respond ONLY with valid JSON. No other text.
"""

ARCHITECT_PROMPT = """
You are a Software Architect. Given the PRD below, produce a
System Design document in EXACTLY this JSON format:

{{
    "implementation_approach": "<describe tech stack and approach>",
    "file_list": ["file1.py", "file2.py"],
    "data_structures": "<class definitions with attributes and methods>",
    "interface_definitions": "<interface specs between modules>",
    "program_call_flow": "<describe the sequence of function calls>"
}}

Product Requirement Document:
{prd}

Respond ONLY with valid JSON. No other text.
"""

PROJECT_MANAGER_PROMPT = """
You are a Project Manager. Given the System Design below, break it
down into tasks in EXACTLY this JSON format:

{{
    "required_packages": ["package1", "package2"],
    "task_list": ["file1.py", "file2.py"],
    "shared_knowledge": "<common knowledge all engineers need>",
    "logic_analysis": [
        ["file1.py", "description of what this file does"],
        ["file2.py", "description of what this file does"]
    ]
}}

System Design:
{system_design}

Respond ONLY with valid JSON. No other text.
"""

ENGINEER_PROMPT = """
You are a Software Engineer. Given the task, system design, and PRD below,
write the complete code for the specified file.

Respond in EXACTLY this JSON format:
{{
    "filename": "<filename>",
    "code": "<complete python code>",
    "dependencies": ["dep1", "dep2"]
}}

Task: Write {filename}
File Description: {file_description}

System Design:
{system_design}

PRD:
{prd}

Previously Written Code:
{existing_code}

Respond ONLY with valid JSON. No other text.
"""

QA_ENGINEER_PROMPT = """
You are a QA Engineer. Given the code below, write unit tests
and review the code for issues.

Respond in EXACTLY this JSON format:
{{
    "test_filename": "test_{filename}",
    "test_code": "<complete test code>",
    "review_notes": "<any issues found>"
}}

Code to Test:
{code}

PRD:
{prd}

Respond ONLY with valid JSON. No other text.
"""

# --- LLM Client (HTTP-based, using OpenAI API) ---

class LLMClient:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.url = "https://api.openai.com/v1/chat/completions"

    def call(self, prompt: str) -> str:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        data = {
            "model": "gpt-3.5-turbo",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.2
        }
        response = requests.post(self.url, headers=headers, json=data)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]

print("Prompts and LLM Client (OpenAI) ready.")

Prompts and LLM Client (OpenAI) ready.


---
## Cell 3 — Message Infrastructure (Pub-Sub Core)

This is the **heart of Section 2**. We define:

- **`Message`** — a structured envelope carrying data between agents, with sender, topic, content, and timestamp.
- **`MessageBus`** — a central broker that manages topic-based subscriptions, message delivery, and a full audit trail.

### How it works
```
ProductManager ──publish──▶ [prd_output] ──deliver──▶ Architect (subscribed)
                                                  ──▶ Engineer  (subscribed)
```
Agents never call each other directly — they only interact through the bus.

In [21]:
# ============================================================
# CELL 3: Message & MessageBus — Pub-Sub Infrastructure
# ============================================================

@dataclass
class Message:
    """A structured message envelope for inter-agent communication."""
    id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    sender: str = ""            # Role name of the sending agent
    topic: str = ""             # The topic/channel this message is published to
    content: Any = None         # The payload (dataclass, dict, string, etc.)
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())
    metadata: Dict = field(default_factory=dict)  # Optional extra info

    def summary(self) -> str:
        """Human-readable one-line summary of this message."""
        content_type = type(self.content).__name__
        return f"[{self.timestamp}] {self.sender} -> '{self.topic}' ({content_type}) [id={self.id}]"


class MessageBus:
    """
    Central publish-subscribe message broker.
    
    Agents subscribe to topics they care about.
    When a message is published to a topic, it is stored and
    made available to all subscribers of that topic.
    
    The bus also maintains a full history for audit/debugging.
    """

    def __init__(self):
        # topic -> [list of subscriber names]
        self._subscriptions: Dict[str, List[str]] = {}
        # topic -> [list of Messages]
        self._topic_messages: Dict[str, List[Message]] = {}
        # Full ordered history of all messages
        self._history: List[Message] = []

    def subscribe(self, agent_name: str, topic: str):
        """Register an agent's interest in a topic."""
        if topic not in self._subscriptions:
            self._subscriptions[topic] = []
        if agent_name not in self._subscriptions[topic]:
            self._subscriptions[topic].append(agent_name)
            print(f"    [BUS] '{agent_name}' subscribed to '{topic}'")

    def publish(self, message: Message):
        """Publish a message to its topic. All subscribers can retrieve it."""
        topic = message.topic
        if topic not in self._topic_messages:
            self._topic_messages[topic] = []
        self._topic_messages[topic].append(message)
        self._history.append(message)

        subscriber_count = len(self._subscriptions.get(topic, []))
        print(f"    [BUS] Published: {message.summary()}  "
              f"({subscriber_count} subscriber(s))")

    def get_messages(self, topic: str) -> List[Message]:
        """Retrieve all messages published to a topic."""
        return self._topic_messages.get(topic, [])

    def get_latest(self, topic: str) -> Optional[Message]:
        """Get the most recent message on a topic, or None."""
        msgs = self.get_messages(topic)
        return msgs[-1] if msgs else None

    def get_subscribers(self, topic: str) -> List[str]:
        """List all agents subscribed to a topic."""
        return self._subscriptions.get(topic, [])

    def get_all_topics(self) -> List[str]:
        """List all topics that have been created."""
        return list(set(
            list(self._subscriptions.keys()) +
            list(self._topic_messages.keys())
        ))

    def get_history(self) -> List[Message]:
        """Full ordered audit trail of every message."""
        return self._history

    def print_history(self):
        """Pretty-print the full message history."""
        print("\n" + "=" * 60)
        print("MESSAGE BUS — FULL COMMUNICATION HISTORY")
        print("=" * 60)
        for i, msg in enumerate(self._history, 1):
            print(f"  {i}. {msg.summary()}")
        print(f"\nTotal messages: {len(self._history)}")
        print(f"Active topics: {', '.join(self.get_all_topics())}")
        print("=" * 60)

# Quick smoke test
bus = MessageBus()
bus.subscribe("TestAgent", "test_topic")
bus.publish(Message(sender="System", topic="test_topic", content="hello"))
assert len(bus.get_messages("test_topic")) == 1
assert bus.get_latest("test_topic").content == "hello"
print("\n✅ Message & MessageBus working correctly.")

    [BUS] 'TestAgent' subscribed to 'test_topic'
    [BUS] Published: [2026-03-11T18:21:50.632218] System -> 'test_topic' (str) [id=4fa9c253]  (1 subscriber(s))

✅ Message & MessageBus working correctly.


---
## Cell 4 — PubSubAgent Base Class

Extends the base agent pattern with pub-sub capabilities:
- On init, automatically subscribes to its declared input topics
- `listen(topic)` — pulls messages from the bus
- `announce(topic, content)` — publishes output to the bus
- `run()` — follows a **listen → process → announce** lifecycle

In [22]:
# ============================================================
# CELL 4: PubSubAgent — Base Class with Pub-Sub Capabilities
# ============================================================

class PubSubAgent:
    """
    Base class for all role agents in the pub-sub pipeline.
    
    Each agent declares:
      - input_topics:  topics it subscribes to (reads from)
      - output_topic:  topic it publishes to (writes to)
      - output_schema: expected dataclass for structured output
    
    Lifecycle:  subscribe -> listen -> process (LLM) -> announce
    """

    def __init__(self, llm: LLMClient, bus: MessageBus):
        self.llm = llm
        self.bus = bus
        self.role = "Base"
        self.input_topics: List[str] = []
        self.output_topic: str = ""
        self.output_schema = None

    def register(self):
        """Subscribe to all declared input topics on the bus."""
        for topic in self.input_topics:
            self.bus.subscribe(self.role, topic)
        print(f"  [{self.role}] Registered. "
              f"Listens: {self.input_topics} | Publishes: '{self.output_topic}'")

    def listen(self, topic: str) -> Any:
        """Pull the latest message content from a subscribed topic."""
        msg = self.bus.get_latest(topic)
        if msg is None:
            print(f"  [{self.role}] No messages on '{topic}' yet.")
            return None
        print(f"  [{self.role}] Received from '{topic}': {type(msg.content).__name__}")
        return msg.content

    def announce(self, content: Any, metadata: Dict = None):
        """Publish output to this agent's output topic."""
        msg = Message(
            sender=self.role,
            topic=self.output_topic,
            content=content,
            metadata=metadata or {}
        )
        self.bus.publish(msg)

    def _build_prompt(self, **kwargs) -> str:
        """Subclasses override this to build their role-specific prompt."""
        raise NotImplementedError

    def _parse_response(self, response: str) -> dict:
        """Parse the LLM JSON response, stripping markdown fences."""
        cleaned = response.strip()
        if cleaned.startswith("```json"):
            cleaned = cleaned[7:]
        if cleaned.startswith("```"):
            cleaned = cleaned[3:]
        if cleaned.endswith("```"):
            cleaned = cleaned[:-3]
        return json.loads(cleaned.strip())

    def _validate(self, parsed: dict) -> bool:
        """Validate parsed output. Only checks that it's a dict — missing fields get defaults."""
        if self.output_schema is None:
            return True
        if not isinstance(parsed, dict):
            print(f"    [{self.role}] Response is not a dict")
            return False
        return True

    def _to_dataclass(self, parsed: dict):
        """Convert parsed dict to the output schema dataclass, filling defaults for missing fields."""
        if self.output_schema is None:
            return parsed
        # Only pass keys that the dataclass expects, let defaults handle the rest
        valid_fields = set(self.output_schema.__dataclass_fields__.keys())
        filtered = {k: v for k, v in parsed.items() if k in valid_fields}
        return self.output_schema(**filtered)

    def process(self, max_retries: int = 3, **kwargs):
        """
        Core processing: build prompt -> call LLM -> parse -> validate.
        Returns the structured dataclass output.
        """
        prompt = self._build_prompt(**kwargs)
        for attempt in range(max_retries):
            print(f"  [{self.role}] Attempt {attempt + 1}/{max_retries}...")
            try:
                response = self.llm.call(prompt)
                parsed = self._parse_response(response)
                if self._validate(parsed):
                    print(f"  [{self.role}] ✅ Processing succeeded!")
                    return self._to_dataclass(parsed)
                else:
                    print(f"  [{self.role}] Validation failed, retrying...")
            except json.JSONDecodeError as e:
                print(f"  [{self.role}] JSON error: {e}, retrying...")
            except Exception as e:
                print(f"  [{self.role}] Error: {e}, retrying...")
        raise RuntimeError(f"[{self.role}] Failed after {max_retries} attempts")

    def run(self, **kwargs):
        """
        Full pub-sub lifecycle:
          1. Listen  — gather inputs from subscribed topics
          2. Process — call LLM with role-specific prompt
          3. Announce — publish output to the bus
        Subclasses override this to define their specific listen/announce flow.
        """
        raise NotImplementedError("Subclasses must implement run()")

print("PubSubAgent base class ready.")


PubSubAgent base class ready.


---
## Cell 5 — Role Agents (Pub-Sub Version)

Each role agent implements the **listen → process → announce** pattern:

1. **ProductManager** — Listens for `user_requirements`, publishes PRD to `prd_output`
2. **Architect** — Listens for `prd_output`, publishes design to `system_design`
3. **ProjectManager** — Listens for `system_design`, publishes tasks to `task_breakdown`
4. **Engineer** — Listens for `task_breakdown` + `prd_output` + `system_design`, publishes code to `code_output`
5. **QAEngineer** — Listens for `code_output` + `prd_output`, publishes tests to `test_output`

In [23]:
# ============================================================
# CELL 5: All Role Agents — Pub-Sub Version
# ============================================================

class PubSubProductManager(PubSubAgent):
    """Listens: user_requirements | Publishes: prd_output"""

    def __init__(self, llm, bus):
        super().__init__(llm, bus)
        self.role = "Product Manager"
        self.input_topics = ["user_requirements"]
        self.output_topic = "prd_output"
        self.output_schema = PRDDocument

    def _build_prompt(self, user_requirement="", **kw):
        return PRODUCT_MANAGER_PROMPT.format(user_requirement=user_requirement)

    def run(self, **kwargs):
        # 1. LISTEN — get user requirement from bus
        user_req = self.listen("user_requirements")
        # 2. PROCESS — call LLM to produce PRD
        prd = self.process(user_requirement=user_req)
        # 3. ANNOUNCE — publish PRD to the bus
        self.announce(prd, metadata={"stage": 1, "description": "Product Requirement Document"})
        return prd


class PubSubArchitect(PubSubAgent):
    """Listens: prd_output | Publishes: system_design"""

    def __init__(self, llm, bus):
        super().__init__(llm, bus)
        self.role = "Architect"
        self.input_topics = ["prd_output"]
        self.output_topic = "system_design"
        self.output_schema = SystemDesign

    def _build_prompt(self, prd=None, **kw):
        return ARCHITECT_PROMPT.format(prd=json.dumps(prd.__dict__, indent=2))

    def run(self, **kwargs):
        # 1. LISTEN — get PRD from bus
        prd = self.listen("prd_output")
        # 2. PROCESS — call LLM to produce system design
        design = self.process(prd=prd)
        # 3. ANNOUNCE — publish design to the bus
        self.announce(design, metadata={"stage": 2, "description": "System Design Document"})
        return design


class PubSubProjectManager(PubSubAgent):
    """Listens: system_design | Publishes: task_breakdown"""

    def __init__(self, llm, bus):
        super().__init__(llm, bus)
        self.role = "Project Manager"
        self.input_topics = ["system_design"]
        self.output_topic = "task_breakdown"
        self.output_schema = TaskList

    def _build_prompt(self, system_design=None, **kw):
        return PROJECT_MANAGER_PROMPT.format(
            system_design=json.dumps(system_design.__dict__, indent=2)
        )

    def run(self, **kwargs):
        # 1. LISTEN — get system design from bus
        design = self.listen("system_design")
        # 2. PROCESS — call LLM to break into tasks
        tasks = self.process(system_design=design)
        # 3. ANNOUNCE — publish task list to the bus
        self.announce(tasks, metadata={"stage": 3, "description": "Task Breakdown"})
        return tasks


class PubSubEngineer(PubSubAgent):
    """Listens: task_breakdown, prd_output, system_design | Publishes: code_output"""

    def __init__(self, llm, bus):
        super().__init__(llm, bus)
        self.role = "Engineer"
        self.input_topics = ["task_breakdown", "prd_output", "system_design"]
        self.output_topic = "code_output"
        self.output_schema = CodeOutput

    def _build_prompt(self, filename="", file_description="",
                      system_design=None, prd=None, existing_code=None, **kw):
        return ENGINEER_PROMPT.format(
            filename=filename,
            file_description=file_description,
            system_design=json.dumps(system_design.__dict__, indent=2),
            prd=json.dumps(prd.__dict__, indent=2),
            existing_code=json.dumps(existing_code or {}, indent=2)
        )

    def run(self, **kwargs):
        # 1. LISTEN — gather inputs from multiple topics
        tasks = self.listen("task_breakdown")
        prd = self.listen("prd_output")
        design = self.listen("system_design")

        # 2. PROCESS — write code for each file in the task list
        code_files = {}
        for filename, desc in tasks.logic_analysis:
            print(f"\n--- [{self.role}] Writing: {filename} ---")
            code = self.process(
                filename=filename,
                file_description=desc,
                system_design=design,
                prd=prd,
                existing_code={k: v.code for k, v in code_files.items()}
            )
            code_files[filename] = code

            # 3. ANNOUNCE — publish each file to the bus individually
            self.announce(code, metadata={
                "stage": 4,
                "filename": filename,
                "description": f"Code for {filename}"
            })

        return code_files


class PubSubQAEngineer(PubSubAgent):
    """Listens: code_output, prd_output | Publishes: test_output"""

    def __init__(self, llm, bus):
        super().__init__(llm, bus)
        self.role = "QA Engineer"
        self.input_topics = ["code_output", "prd_output"]
        self.output_topic = "test_output"
        self.output_schema = TestOutput

    def _build_prompt(self, code=None, prd=None, **kw):
        return QA_ENGINEER_PROMPT.format(
            filename=code.filename,
            code=code.code,
            prd=json.dumps(prd.__dict__, indent=2)
        )

    def run(self, code_files: dict, **kwargs):
        # 1. LISTEN — get PRD from bus (code_files passed directly since
        #    there are multiple code messages on the bus)
        prd = self.listen("prd_output")

        # 2. PROCESS — write tests for each code file
        test_files = {}
        for fname, code_obj in code_files.items():
            print(f"\n--- [{self.role}] Testing: {fname} ---")
            test = self.process(code=code_obj, prd=prd)
            test_files[fname] = test

            # 3. ANNOUNCE — publish each test to the bus
            self.announce(test, metadata={
                "stage": 5,
                "filename": f"test_{fname}",
                "description": f"Tests for {fname}"
            })

        return test_files


print("All pub-sub role agents defined.")

All pub-sub role agents defined.


---
## Cell 6 — PubSubPipeline Orchestrator

The orchestrator:
1. Creates the shared `MessageBus`
2. Initializes all agents and registers their subscriptions
3. Seeds the bus with the initial user requirement
4. Triggers each agent in dependency order (each agent pulls its own inputs from the bus)
5. Prints the full communication history at the end

In [24]:
# ============================================================
# CELL 6: PubSubPipeline — Orchestrator
# ============================================================

class PubSubPipeline:
    """
    Orchestrates the multi-agent software development pipeline
    using publish-subscribe communication.
    
    Key difference from the SOP pipeline (Section 1):
    - Agents don't receive data as function arguments
    - Instead they pull inputs from the MessageBus topics
    - Outputs are published back to the bus for downstream agents
    - Full audit trail of all inter-agent communications
    """

    def __init__(self, llm: LLMClient):
        # Create the shared message bus
        self.bus = MessageBus()

        # Initialize all agents with the shared bus
        self.pm = PubSubProductManager(llm, self.bus)
        self.architect = PubSubArchitect(llm, self.bus)
        self.proj_mgr = PubSubProjectManager(llm, self.bus)
        self.engineer = PubSubEngineer(llm, self.bus)
        self.qa = PubSubQAEngineer(llm, self.bus)

        # Register all subscriptions
        print("\n📡 REGISTERING AGENT SUBSCRIPTIONS")
        print("-" * 40)
        self.pm.register()
        self.architect.register()
        self.proj_mgr.register()
        self.engineer.register()
        self.qa.register()
        print("-" * 40)
        print(f"Active topics: {self.bus.get_all_topics()}\n")

    def run(self, user_requirement: str) -> dict:
        """
        Execute the full pipeline:
          1. Seed the bus with user requirement
          2. Trigger agents in dependency order
          3. Each agent listens, processes, and announces via the bus
        """

        # --- Seed the bus with the user requirement ---
        print("=" * 60)
        print("🚀 SEEDING MESSAGE BUS WITH USER REQUIREMENT")
        print("=" * 60)
        seed_msg = Message(
            sender="User",
            topic="user_requirements",
            content=user_requirement,
            metadata={"stage": 0, "description": "Initial user requirement"}
        )
        self.bus.publish(seed_msg)

        # --- Stage 1: Product Manager ---
        print("\n" + "=" * 60)
        print("📋 STAGE 1: Product Manager — listen → process → announce")
        print("=" * 60)
        prd = self.pm.run()

        # --- Stage 2: Architect ---
        print("\n" + "=" * 60)
        print("🏗️  STAGE 2: Architect — listen → process → announce")
        print("=" * 60)
        design = self.architect.run()

        # --- Stage 3: Project Manager ---
        print("\n" + "=" * 60)
        print("📊 STAGE 3: Project Manager — listen → process → announce")
        print("=" * 60)
        tasks = self.proj_mgr.run()

        # --- Stage 4: Engineer ---
        print("\n" + "=" * 60)
        print("💻 STAGE 4: Engineer — listen → process → announce")
        print("=" * 60)
        code_files = self.engineer.run()

        # --- Stage 5: QA Engineer ---
        print("\n" + "=" * 60)
        print("🧪 STAGE 5: QA Engineer — listen → process → announce")
        print("=" * 60)
        test_files = self.qa.run(code_files=code_files)

        # --- Print full communication history ---
        self.bus.print_history()

        return {
            "prd": prd,
            "system_design": design,
            "task_list": tasks,
            "code": code_files,
            "tests": test_files,
            "message_history": self.bus.get_history()
        }

print("PubSubPipeline orchestrator ready.")

PubSubPipeline orchestrator ready.


---
## Cell 7 — Run the Pipeline
Execute the full pub-sub pipeline with a sample requirement.

In [25]:
# ============================================================
# CELL 7: RUN THE PUB-SUB PIPELINE
# ============================================================

llm = LLMClient(api_key=API_KEY)
pipeline = PubSubPipeline(llm)

result = pipeline.run("Create a classic and simple Flappy Bird game")


📡 REGISTERING AGENT SUBSCRIPTIONS
----------------------------------------
    [BUS] 'Product Manager' subscribed to 'user_requirements'
  [Product Manager] Registered. Listens: ['user_requirements'] | Publishes: 'prd_output'
    [BUS] 'Architect' subscribed to 'prd_output'
  [Architect] Registered. Listens: ['prd_output'] | Publishes: 'system_design'
    [BUS] 'Project Manager' subscribed to 'system_design'
  [Project Manager] Registered. Listens: ['system_design'] | Publishes: 'task_breakdown'
    [BUS] 'Engineer' subscribed to 'task_breakdown'
    [BUS] 'Engineer' subscribed to 'prd_output'
    [BUS] 'Engineer' subscribed to 'system_design'
  [Engineer] Registered. Listens: ['task_breakdown', 'prd_output', 'system_design'] | Publishes: 'code_output'
    [BUS] 'QA Engineer' subscribed to 'code_output'
    [BUS] 'QA Engineer' subscribed to 'prd_output'
  [QA Engineer] Registered. Listens: ['code_output', 'prd_output'] | Publishes: 'test_output'
---------------------------------------

RuntimeError: [Product Manager] Failed after 3 attempts

---
## Cell 8 — View PRD Output

In [ ]:
# ============================================================
# CELL 8: View PRD Output
# ============================================================

print("PRODUCT GOALS:")
for g in result["prd"].product_goals:
    print(f"  - {g}")

print("\nUSER STORIES:")
for s in result["prd"].user_stories:
    print(f"  - {s}")

print("\nREQUIREMENT POOL:")
for req, priority in result["prd"].requirement_pool:
    print(f"  [{priority}] {req}")

PRODUCT GOALS:
  - Engage users with a fun and addictive gameplay
  - Provide a nostalgic experience for fans of the original game
  - Ensure smooth and responsive controls

USER STORIES:
  - As a player, I want to be able to control the bird by tapping the screen to avoid obstacles
  - As a player, I want to see my score displayed on the screen as I progress through the game
  - As a player, I want the game to have simple graphics and animations

REQUIREMENT POOL:
  [P0] Implement tap control for the bird's movement
  [P1] Display the player's score on the screen


---
## Cell 9 — View System Design

In [ ]:
# ============================================================
# CELL 9: View System Design
# ============================================================

print("IMPLEMENTATION APPROACH:")
print(f"  {result['system_design'].implementation_approach}")

print("\nFILE LIST:")
for f in result["system_design"].file_list:
    print(f"  - {f}")

print("\nDATA STRUCTURES:")
print(f"  {result['system_design'].data_structures}")

IMPLEMENTATION APPROACH:
  The game will be implemented using HTML5, CSS, and JavaScript. The game logic will be handled in JavaScript, while CSS will be used for styling the UI. The game will be designed to run in a web browser.

FILE LIST:
  - game.js
  - style.css

DATA STRUCTURES:
  class Bird { attributes: position, velocity, alive methods: flap(), update(), draw() } class Pipe { attributes: position, gap, width methods: update(), draw() }


---
## Cell 10 — View Task Breakdown

In [ ]:
# ============================================================
# CELL 10: View Task Breakdown
# ============================================================

print("TASKS:")
for fname, desc in result["task_list"].logic_analysis:
    print(f"  {fname}: {desc}")

print("\nREQUIRED PACKAGES:")
for pkg in result["task_list"].required_packages:
    print(f"  - {pkg}")

TASKS:
  game.js: Handles game logic in JavaScript
  style.css: Styles the UI of the game

REQUIRED PACKAGES:
  - HTML5
  - CSS
  - JavaScript


---
## Cell 11 — View Generated Code

In [ ]:
# ============================================================
# CELL 11: View Generated Code
# ============================================================

for fname, code_obj in result["code"].items():
    print(f"\n{'=' * 50}")
    print(f"FILE: {fname}")
    print(f"{'=' * 50}")
    print(code_obj.code)


FILE: game.js
const canvas = document.getElementById('gameCanvas');
const ctx = canvas.getContext('2d');

// Game logic implementation here

function initializeGame() {
    // Initialize game elements and event listeners
}

function startGameLoop() {
    // Start game loop
}

function updateBird() {
    // Update bird position
}

function updatePipes() {
    // Update pipes position
}

function checkCollisions() {
    // Check for collisions
}

function updateScore() {
    // Update score and display on screen
}

initializeGame();
startGameLoop();

FILE: style.css
/* Styles for the UI of the game */

body {
    margin: 0;
    padding: 0;
    overflow: hidden;
}

canvas {
    display: block;
    margin: 0 auto;
    background-color: #70c5ce;
}

.bird {
    width: 40px;
    height: 40px;
    background-color: yellow;
    border-radius: 50%;
}

.pipe {
    position: absolute;
    width: 80px;
    height: 300px;
    background-color: green;
}

.score {
    position: absolute;
    top: 10p

---
## Cell 12 — View Tests and Reviews

In [ ]:
# ============================================================
# CELL 12: View Tests and Reviews
# ============================================================

for fname, test_obj in result["tests"].items():
    print(f"\n{'=' * 50}")
    print(f"TEST FOR: {fname}")
    print(f"{'=' * 50}")
    print(test_obj.test_code)
    print(f"\nReview: {test_obj.review_notes}")


TEST FOR: game.js
describe('Flappy Bird Game', () => {
    beforeEach(() => {
        // Setup canvas and game elements
    });

    it('should update bird position correctly', () => {
        // Test updateBird function
    });

    it('should update pipes position correctly', () => {
        // Test updatePipes function
    });

    it('should check collisions properly', () => {
        // Test checkCollisions function
    });

    it('should update score correctly', () => {
        // Test updateScore function
    });
});

Review: The code provided is missing the actual game logic implementation, making it difficult to write meaningful unit tests. It would be helpful to have the game logic in place to properly test the functions. Additionally, the code lacks error handling and input validation, which could lead to unexpected behavior during gameplay.

TEST FOR: style.css
describe('UI Styles', () => {
  test('body styles', () => {
    expect(document.body.style.margin).toBe('0');
  

---
## Cell 13 — Communication History (Pub-Sub Audit Trail)

This cell shows the **full audit trail** of all messages exchanged between agents via the message bus — the key differentiator of the pub-sub approach.

In [ ]:
# ============================================================
# CELL 13: Full Message Bus Communication History
# ============================================================

print("\n📡 PUBLISH-SUBSCRIBE COMMUNICATION AUDIT TRAIL")
print("=" * 65)

for i, msg in enumerate(result["message_history"], 1):
    stage = msg.metadata.get("stage", "?")
    desc = msg.metadata.get("description", "")
    content_type = type(msg.content).__name__
    print(f"\n  Message #{i}")
    print(f"    ID:        {msg.id}")
    print(f"    Timestamp: {msg.timestamp}")
    print(f"    Sender:    {msg.sender}")
    print(f"    Topic:     {msg.topic}")
    print(f"    Payload:   {content_type}")
    print(f"    Stage:     {stage}")
    if desc:
        print(f"    Info:      {desc}")

print(f"\n{'=' * 65}")
print(f"Total messages exchanged: {len(result['message_history'])}")
print(f"Topics used: {list(set(m.topic for m in result['message_history']))}")
print("\n✅ Pub-Sub pipeline complete — all agents communicated via the MessageBus.")


📡 PUBLISH-SUBSCRIBE COMMUNICATION AUDIT TRAIL

  Message #1
    ID:        1a0eceaf
    Timestamp: 2026-03-11T17:49:36.117537
    Sender:    User
    Topic:     user_requirements
    Payload:   str
    Stage:     0
    Info:      Initial user requirement

  Message #2
    ID:        4319a71a
    Timestamp: 2026-03-11T17:49:37.804696
    Sender:    Product Manager
    Topic:     prd_output
    Payload:   PRDDocument
    Stage:     1
    Info:      Product Requirement Document

  Message #3
    ID:        76eb98c3
    Timestamp: 2026-03-11T17:49:39.586755
    Sender:    Architect
    Topic:     system_design
    Payload:   SystemDesign
    Stage:     2
    Info:      System Design Document

  Message #4
    ID:        a49b85dd
    Timestamp: 2026-03-11T17:49:40.641162
    Sender:    Project Manager
    Topic:     task_breakdown
    Payload:   TaskList
    Stage:     3
    Info:      Task Breakdown

  Message #5
    ID:        60523c03
    Timestamp: 2026-03-11T17:49:42.740732
    Sender